# Problem 9: Build a Database Query Tool

**Difficulty:** Easy | **Framework:** LangChain

**Categories:** tool-calling

This notebook accompanies [Agent Foundry](https://agent-foundry-pi.vercel.app/problems/build-a-database-query-tool).

## 1. Install Dependencies

In [ ]:
!pip install -q langchain langchain-openai langchain-community langgraph

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- The tool must prevent SQL injection attacks
- Only SELECT queries are allowed
- Results must be formatted as a readable string, not raw tuples


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
import sqlite3

llm = ChatOpenAI(model="gpt-4o-mini")

# Set up in-memory database with sample data
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
cursor.execute("CREATE TABLE employees (id INTEGER, name TEXT, department TEXT, salary INTEGER)")
cursor.executemany("INSERT INTO employees VALUES (?, ?, ?, ?)", [
    (1, "Alice", "Engineering", 95000),
    (2, "Bob", "Marketing", 72000),
    (3, "Charlie", "Engineering", 88000),
    (4, "Diana", "Sales", 67000),
    (5, "Eve", "Marketing", 78000),
])
conn.commit()

# BUG: This tool has no SQL injection protection, no query validation,
# and returns raw tuples instead of formatted results
@tool
def query_database(sql: str) -> str:
    """Run a SQL query against the employees database."""
    result = cursor.execute(sql).fetchall()
    return str(result)

tools = [query_database]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a database assistant. Use the query tool to answer questions about employees."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools)

result = executor.invoke({"input": "Show me all engineers and their salaries"})
print(result["output"])

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. Use parameterized queries or whitelist allowed table names to prevent injection
2. Check that the query starts with SELECT before executing it
3. Format results with column headers and handle the case where no rows are returned


## 7. Evaluation Criteria

Check your solution against these criteria:

- SQL injection prevented
- Results formatted
- Handles empty results
